# L08: A network you can watch train

**Day 2.** A one-hidden-layer network in NumPy: forward pass, backpropagation as the chain rule, why the nonlinear activation is mandatory, parameters converted to memory, and scaled dot-product attention, the core of the transformer.

- Reads: checkpoints/L02_clean.csv, checkpoints/L04_split.csv, data/token_vectors.json
- Writes: work/L08_results.json

**Before you start.** Open this notebook from the **Notebooks** folder of the class workspace, next to the **data** and **checkpoints** folders, and run the first two cells.

**How this lab runs.** Work through the cells in order on your own. Each one says what to do with it: **RUN** supplied code, **PREDICT** before running, **DECIDE** by filling a blank, or **WRITE** an answer in words. When most of the room is done, we discuss the answers as a class, then walk through the worked solution together.

### Step 1 · RUN

In [ ]:
# RUN · setup: finds the class files and checks that today's are there. Run it first, every time.
from pathlib import Path
# The class files sit in the folder holding "data" and "checkpoints": this notebook's folder or the one above it.
# (On a local install, a folder named AIDSA in your user folder also works.)
_here = Path.cwd()
ROOT = next((p for p in [_here, *_here.parents, Path.home() / "AIDSA"] if (p / "data").is_dir() and (p / "checkpoints").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Class files not found: open this notebook from the Notebooks folder of the class workspace.")
DATA, CHECKPOINTS, WORK = ROOT / "data", ROOT / "checkpoints", ROOT / "work"
NEEDED = ["checkpoints/L02_clean.csv",
          "checkpoints/L04_split.csv",
          "checkpoints/L06_choice.json",
          "data/token_vectors.json"]
missing = [name for name in NEEDED if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError("Not in the workspace yet: " + ", ".join(missing) + ". Ask your instructor: today's files are not in the workspace yet.")
WORK.mkdir(exist_ok=True)
SOURCE = "checkpoint"  # change to "mine" to use your own saved work where it exists
SEED = 2026
import numpy as np
rng = np.random.default_rng(SEED)

def input_path(name):
    """Where this lab reads an earlier lab's output: your own work folder if you chose "mine" and the file is there,
    otherwise the supplied checkpoint."""
    mine = WORK / name
    return mine if SOURCE == "mine" and mine.exists() else CHECKPOINTS / name

print("Found the class files in:", ROOT, "| start from:", SOURCE)

### Step 2 · RUN

In [ ]:
# RUN · helper functions: the tested course helpers this lab uses. Run it once, after the setup cell.
# You do not need to read them to do the lab; each is explained where the lab uses it.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


ACTIVATIONS = {
    "relu": (lambda z: np.maximum(z, 0.0), lambda z: (z > 0).astype(float)),
    "none": (lambda z: z, lambda z: np.ones_like(z)),
}


def sigmoid(z):
    """Logistic function: maps any score to a probability between 0 and 1.

    Lab L08.
    """
    return 1.0 / (1.0 + np.exp(-z))


def init_params(n_inputs, n_hidden, seed=2026):
    """Random starting weights for a one-hidden-layer network (seeded).

    Lab L08.
    """
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, 1, (n_inputs, n_hidden)), "b1": rng.normal(0, 1, n_hidden),
            "W2": rng.normal(0, 1 / np.sqrt(n_hidden), (n_hidden, 1)), "b2": np.zeros(1)}


def forward(X, params, activation="relu"):
    """Forward pass. Returns (h_pre, h, p): hidden pre-activation, hidden output, predicted probability.

    Lab L08.
    """
    act = ACTIVATIONS[activation][0]
    h_pre = X @ params["W1"] + params["b1"]
    h = act(h_pre)
    p = sigmoid(h @ params["W2"] + params["b2"])
    return h_pre, h, p


def param_count(arrays):
    """Total number of parameters in a collection of weight arrays (a dict or a list).

    Lab L08.
    """
    values = arrays.values() if isinstance(arrays, dict) else arrays
    return int(sum(np.asarray(a).size for a in values))


def memory_bytes(n_params, bits=32):
    """Memory to hold n_params parameters at the given precision (32, 16, 8 or 4 bits), in bytes.

    Weights only: training needs several times more for gradients and optimizer state.

    Lab L08.
    """
    return n_params * bits / 8


def softmax(scores, axis=-1):
    """Softmax along an axis, stabilised by subtracting the maximum.

    Parameters
    ----------
    scores : array-like
        Raw scores.
    axis : int
        Axis along which the result sums to one (default: last).

    Returns
    -------
    numpy.ndarray of the same shape, non-negative, summing to 1 along `axis`.
    """
    scores = np.asarray(scores, dtype=float)
    shifted = scores - scores.max(axis=axis, keepdims=True)
    exps = np.exp(shifted)
    return exps / exps.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """Scaled dot-product attention: softmax(Q K^T / sqrt(d_k)) V.

    Parameters
    ----------
    Q, K : array (n_tokens, d_k)
        Queries and keys.
    V : array (n_tokens, d_v)
        Values.

    Returns
    -------
    (output, weights): output is (n_tokens, d_v); weights is (n_tokens, n_tokens),
    each row summing to 1. Row i says how much token i draws on every token.

    Lab L08, part 5.
    """
    Q, K, V = (np.asarray(a, dtype=float) for a in (Q, K, V))
    d_k = Q.shape[1]
    weights = softmax(Q @ K.T / np.sqrt(d_k), axis=1)
    return weights @ V, weights


NUMERIC = ["aircraft_age_years", "total_flight_hours", "days_since_inspection", "ground_time_hours",
           "cargo_weight_klb", "temperature_c", "sortie_duration_hours", "open_major", "open_minor"]


CATEGORICAL = ["aircraft_type", "home_base", "mission_type"]


def build_preprocessor(numeric=NUMERIC, categorical=CATEGORICAL):
    """Unfitted preprocessing: numeric median imputation with missing indicators, then scaling;
    categorical one-hot encoding that ignores unseen levels.

    Fit it on training rows only.

    Lab L04.
    """
    numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                              ("scale", StandardScaler())])
    return ColumnTransformer([
        ("num", numeric_steps, numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])

print("Helper functions ready")

## Part 1: Forward pass

**Specification.** Build a network with one input (temperature at departure), eight hidden units and one output (the
probability of a maintenance delay), and run one forward pass with random weights.

### Step 3 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.read_csv(input_path("L02_clean.csv"), encoding="utf-8")
df = df.merge(pd.read_csv(input_path("L04_split.csv"), encoding="utf-8"), on="mission_id", validate="one_to_one")
train, test = df[df["split"] == "train"], df[df["split"] == "test"]

# Standardise temperature with the training rows' mean and spread (the train-only rule from L04).
mu, sd = train["temperature_c"].mean(), train["temperature_c"].std()
X_train = ((train[["temperature_c"]] - mu) / sd).to_numpy()
X_test = ((test[["temperature_c"]] - mu) / sd).to_numpy()
y_train = train[["maint_delay"]].to_numpy()
y_test = test[["maint_delay"]].to_numpy()

relu = lambda z: np.maximum(z, 0.0)            # the activation: zero below 0, identity above
sigmoid = lambda z: 1 / (1 + np.exp(-z))        # turns the output score into a probability

params = init_params(n_inputs=1, n_hidden=8, seed=SEED)
h_pre = X_train @ params["W1"] + params["b1"]   # 8 weighted sums per departure
h = relu(h_pre)                                 # 8 hidden activations
p = sigmoid(h @ params["W2"] + params["b2"])    # one probability per departure
loss = -np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))
print(f"shapes: X {X_train.shape}, hidden {h.shape}, output {p.shape}")
print(f"untrained: mean predicted risk {p.mean():.3f} against an actual rate of {y_train.mean():.3f}; loss {loss:.3f}")

### Step 4 · WRITE

**WRITE · your interpretation**. Trace one departure through the network: what happens at the hidden layer and at the output, and what
does the loss measure?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 2: Backpropagation

**Specification.** Read the backward pass (the chain rule, layer by layer), check its gradient against a numerical estimate,
then train for 400 epochs and watch the loss fall.

### Step 5 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
def backward(X, y, params, h_pre, h, p, act_grad):
    """Gradients of the mean cross-entropy loss with respect to every weight, by the chain rule."""
    dout = (p - y) / len(X)                     # gradient at the output score (sigmoid with cross-entropy)
    dW2 = h.T @ dout                            # output weights: hidden activation times output gradient
    db2 = dout.sum(axis=0)
    dh = (dout @ params["W2"].T) * act_grad(h_pre)   # hidden layer: back through W2, then the activation's slope
    dW1 = X.T @ dh
    db1 = dh.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

### Step 6 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# Verify against a known result: nudge one hidden weight up and down and measure the change in loss.
def loss_of(params, activation):
    _, _, p = forward(X_train, params, activation)
    return float(-np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p)))


relu_grad = ACTIVATIONS["relu"][1]
h_pre, h, p = forward(X_train, params, "relu")
grads = backward(X_train, y_train, params, h_pre, h, p, relu_grad)
eps = 1e-5
up = {k: v.copy() for k, v in params.items()}
down = {k: v.copy() for k, v in params.items()}
up["W1"][0, 3] += eps
down["W1"][0, 3] -= eps
numeric = (loss_of(up, "relu") - loss_of(down, "relu")) / (2 * eps)
print(f"hidden weight W1[0,3]: your gradient {grads['W1'][0, 3]:.6f}, numerical estimate {numeric:.6f}")


def fit_network(activation, epochs=400, learning_rate=0.5):
    """Full-batch gradient descent with your backward function (bounded: 400 epochs)."""
    act_grad = ACTIVATIONS[activation][1]
    w = init_params(1, 8, seed=SEED)
    losses = []
    for _ in range(epochs):
        h_pre, h, p = forward(X_train, w, activation)
        losses.append(float(-np.mean(y_train * np.log(p) + (1 - y_train) * np.log(1 - p))))
        g = backward(X_train, y_train, w, h_pre, h, p, act_grad)
        w = {k: w[k] - learning_rate * g[k] for k in w}
    return w, losses


trained, losses = fit_network("relu")
test_auc = roc_auc_score(y_test, forward(X_test, trained, "relu")[2])
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}; test AUC {test_auc:.3f}")

temps = np.linspace(-25, 42, 100)
risk = forward(((temps - mu) / sd)[:, None], trained, "relu")[2].ravel()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.plot(losses)
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax2.plot(temps, risk)
ax2.set_xlabel("temperature (C)")
ax2.set_ylabel("predicted delay risk")
plt.tight_layout()
plt.show()

### Step 7 · WRITE

**WRITE · your interpretation**. What shape has the network learned, and why did we check the gradient numerically before training?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 3: Remove the activation

**Specification.** Retrain the same network, with the same data, epochs and learning rate, but with no activation.

### Step 8 · PREDICT

**PREDICT · before you run the next cell**. Without the activation, can the network still learn the U? Why?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 9 · DECIDE

In [ ]:
# DECIDE · DP-L08-1 · which activation the retrained network uses.
# Replace ____ with your choice. The choices are taught on the slide "Why the activation matters: without it, depth collapses".
# Retrain the same network without its activation, to see what the activation contributes.
ACTIVATION = ____
flat, flat_losses = fit_network(ACTIVATION)
flat_auc = roc_auc_score(y_test, forward(X_test, flat, ACTIVATION)[2])
flat_risk = forward(((temps - mu) / sd)[:, None], flat, ACTIVATION)[2].ravel()
print(f"activation {ACTIVATION!r}: loss {flat_losses[-1]:.3f}, test AUC {flat_auc:.3f} (the original ReLU network: {test_auc:.3f})")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(temps, risk, label="ReLU")
ax.plot(temps, flat_risk, label=f"activation {ACTIVATION!r}")
ax.set_xlabel("temperature (C)")
ax.set_ylabel("predicted delay risk")
ax.legend()
plt.show()

### Step 10 · WRITE

**WRITE · your interpretation**. Describe the failure. Why does every deep network need a nonlinear activation?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 4: Size and memory

**Specification.** Count this network's parameters and convert the count to memory. Then train scikit-learn's
MLPClassifier on all the Day 1 features and compare it with the Day 2 model.

### Step 11 · DECIDE

In [ ]:
# DECIDE · DP-L08-2 · how many bytes each parameter of this network takes.
# Replace ____ with your choice. The choices are taught on the slide "Parameters are memory: why size is a budget".
# This network was trained in 32-bit floating point. Set the bytes each parameter takes.
BYTES_PER_PARAMETER = ____
n_params = trained["W1"].size + trained["b1"].size + trained["W2"].size + trained["b2"].size
memory = n_params * BYTES_PER_PARAMETER
print(f"our network: {n_params} parameters, {memory:.0f} bytes at {BYTES_PER_PARAMETER * 8}-bit")

### Step 12 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

cols = NUMERIC + CATEGORICAL
mlp = Pipeline([("prep", build_preprocessor()),
                ("model", MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, early_stopping=True,
                                        random_state=SEED))])
mlp.fit(train[cols], train["maint_delay"])
mlp_auc = roc_auc_score(test["maint_delay"], mlp.predict_proba(test[cols])[:, 1])
mlp_params = param_count(mlp.named_steps["model"].coefs_ + mlp.named_steps["model"].intercepts_)
day2 = json.loads(input_path("L06_choice.json").read_text(encoding="utf-8"))
print(f"MLP (32, 16) on all features: {mlp_params:,} parameters, test AUC {mlp_auc:.3f}; "
      f"Day 2 {day2['model']}: test AUC {day2['test_auc']:.3f}")

# Pre-computed for models far too large to run here: weights only, before any training overhead.
sizes = {"our network": n_params, "MLP above": mlp_params, "7 billion": 7e9, "70 billion": 70e9}
table = pd.DataFrame({f"{b}-bit": {name: memory_bytes(n, b) / 1e9 for name, n in sizes.items()}
                      for b in (32, 16, 8, 4)})
print("\nweights in gigabytes:")
print(table.map(lambda v: f"{v:,.3g}"))

### Step 13 · WRITE

**WRITE · your interpretation**. Why can a 70-billion-parameter model not run in the class workspace, and what does 4-bit
quantization buy? Did the MLP beat the Day 2 model?

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

## Part 5: Attention

**Specification.** Compute scaled dot-product attention for three tokens from a maintenance report:
*hydraulic*, *leak* and *delay*. Produce the 3 by 3 matrix of attention weights (each row sums to 1)
and the three output vectors. This is the mechanism inside every transformer block, and the
prerequisite for Thursday's language-model and retrieval lectures.

### Step 14 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
import json
tv = json.loads((DATA / "token_vectors.json").read_text(encoding="utf-8"))
tokens = tv["tokens"]
X = np.array(tv["embeddings"])            # one row per token: 3 tokens, 4 dimensions
W_Q, W_K, W_V = (np.array(tv[k]) for k in ("W_Q", "W_K", "W_V"))

Q, K, V = X @ W_Q, X @ W_K, X @ W_V         # project each token into a query, a key and a value
d_k = Q.shape[1]
print("tokens:", tokens)
print("Q", Q.shape, "K", K.shape, "V", V.shape, "d_k =", d_k)


def softmax_rows(scores):
    """Turn each row of scores into weights that are positive and sum to 1."""
    e = np.exp(scores - scores.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

### Step 15 · PREDICT

**PREDICT · before you run the next cell**. Before running the next cell: which token do you expect *leak* to attend to most, itself or another?
Write your guess and one reason.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 16 · DECIDE

In [ ]:
# DECIDE · DP-L08-3 · what the attention scores are divided by.
# Replace ____ with your choice. The choices are taught on the slide "Attention by hand: query, key, value, and the scaling".
# Scores are dot products of queries and keys. Choose what they are divided by before the softmax.
SCALING = ____
divisor = {"sqrt_dk": np.sqrt(d_k), "d_k": float(d_k), "none": 1.0}[SCALING]
scores = Q @ K.T / divisor
weights = softmax_rows(scores)
output = weights @ V

np.set_printoptions(precision=3, suppress=True)
print("attention weights (row = token attending, column = token attended to):")
print(weights)
print("row sums:", weights.sum(axis=1))
print("output vectors:")
print(output)

### Step 17 · RUN

In [ ]:
# RUN · supplied code: read it, then run it.
# Self-check against the course helper, then see what the scaling by sqrt(d_k) does.
ref_output, ref_weights = attention(Q, K, V)
print("matches the course helper:", np.allclose(weights, ref_weights) and np.allclose(output, ref_output))

unscaled = softmax_rows(Q @ K.T)
print("leak -> hydraulic, scaled:  ", round(float(weights[1, 0]), 3))
print("leak -> hydraulic, unscaled:", round(float(unscaled[1, 0]), 3))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(4.2, 3.4))
im = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), tokens)
ax.set_yticks(range(3), tokens)
ax.set_xlabel("attended to")
ax.set_ylabel("attending")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{weights[i, j]:.2f}", ha="center", va="center",
                color="white" if weights[i, j] > 0.5 else "black")
ax.set_title("Attention weights")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Step 18 · WRITE

**WRITE · your interpretation**. (1) Which token does *leak* attend to most, and what does that do to its output vector?
(2) Why divide by sqrt(d_k)? Use the scaled and unscaled numbers above.

*Your answer (double-click this cell, type here, then press Shift+Enter):* 

### Step 19 · RUN

In [ ]:
# RUN · saves this part's output to work/.
results = {"attention_weights": weights.round(4).tolist(), "attention_output": output.round(4).tolist(),
           "tokens": tokens}
(WORK / "L08_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("saved", "work/L08_results.json")